# Flux post-processing chain: IRGA75 latent heat flux (2004-2017)

**notebook version**: `1` (29 Oct 2026). Adapted from the IRGA72 notebook `25_FPPC_IRGA72_LE_2016-2026.09`.

## ℹ️ About this notebook

Processes a latent heat flux with the [diive flux post-processing chain](https://diive.readthedocs.io/en/latest/flux_postprocessing.html): outliers (L3.2),
gap-filling (L4.1) and the conversion to evapotranspiration (ET). As in FLUXNET2015 and ONEFlux
(Pastorello et al. 2020), LE gets no storage term (L3.1) and no USTAR filter (L3.3); both levels
run, because the chain needs them. The site, flux, period and all settings are in *Settings*;
*Data sources* links the site documentation, which holds the decisions and periods specific to the
site.

`TEST_RUN = True` runs one year (`TEST_YEAR`) and writes its own output file.

***

## 📇 Data sources

The input `30_FLUX_POST-PROCESSING_CHAIN/IRGA75/01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet` in
the external data folder holds the Level-1 fluxes of the open-path IRGA75 (LI-7500) runs and the
meteo data of the site. The opco correction of that file applies to the CO2 flux only; `LE` is
used as EddyPro calculated it.

Site documentation (CH-LAE): the processing levels, settings, periods removed by hand and known
limitations are on the [CH-LAE flux product site](https://holukas.github.io/dataset_ch-lae_flux_product/FPC.html), Levels 3.1 to 4.2.

## ⚙️ Settings

In [ ]:
# --- Source ------------------------------------------------------------------------------------
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
SOURCEFILE = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet"

# --- Testing -----------------------------------------------------------------------------------
# TEST_RUN = True processes only TEST_YEAR, a quick check of the whole notebook. A test run writes
# its own output file, so it never overwrites the result of a full run.
TEST_RUN = False
TEST_YEAR = 2004
# Years to process: the full calendar years with IRGA75 fluxes, as in the IRGA75 CO2 notebook 05.
# The input also holds 2018, without any IRGA75 flux, and 2019 with only the stand-in run of
# 17 January to 20 March; neither is gap-filled here.
YEARS = [TEST_YEAR] if TEST_RUN else list(range(2004, 2018))

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "25_FPPC_IRGA75_LE_2004-2017"  # same name as this notebook
if TEST_RUN:
    OUTNAME += f"_TEST_{TEST_YEAR}"
OUTPATH = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75"

# --- Site and flux -----------------------------------------------------------------------------
FLUXCOL = 'LE'  # latent heat flux of the open-path LI-7500 (W m-2), without storage term
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389
UTC_OFFSET = 1  # timestamps are UTC+01:00 all year
NIGHTTIME_THRESHOLD = 20  # W m-2 potential radiation below this = nighttime
DAYTIME_ACCEPT_QCF_BELOW = 2  # keep QCF 0 and 1
NIGHTTIME_ACCEPT_QCF_BELOW = 2

# --- Level 2 -----------------------------------------------------------------------------------
# The VM97 test reads H2O_VM97_TEST, the SSITC and SCF tests LE_SSITC_TEST and LE_SCF.
LEVEL2_SETTINGS = {
    # AGC of the LI-7500 (%): 50 is a very good signal, above 90 a bad one, as in the CO2 notebook 05.
    'signal_strength': {'apply': True,
                        'signal_strength_col': 'CUSTOM_AGC_IRGA75_MEAN',
                        'method': 'discard outside',
                        'threshold': (40, 90)},
    'raw_data_screening_vm97': {'apply': True,
                                'spikes': True, 'amplitude': False, 'dropout': True,
                                'abslim': False, 'skewkurt_hf': False, 'skewkurt_sf': False,
                                'discont_hf': False, 'discont_sf': False},
    'ssitc': {'apply': True, 'setflag_timeperiod': None},
    'gas_completeness': {'apply': False},
    'spectral_correction_factor': {'apply': True},
    'angle_of_attack': {'apply': False, 'application_dates': []},
    'steadiness_of_horizontal_wind': {'apply': False},
}
LEVEL2_INPUTS = ['LE_SSITC_TEST', 'LE_SCF', 'H2O_VM97_TEST']

# --- Level 3.2 ---------------------------------------------------------------------------------
# Periods removed by hand (date ranges, whole days, both ends included). The reasons are in the
# site documentation (Level 3.2).
REMOVE_DATES = []  # none for the IRGA75 runs so far
# Absolute limits (W m-2), as in the 2016-2024 LE notebook.
ABSLIM = (-50, 800)

# --- Level 3.3 ---------------------------------------------------------------------------------
# No USTAR filter for LE. The chain needs Level 3.3 before gap-filling, so it runs with a single
# scenario at a threshold of 0, which removes no record.
USTAR_THRESHOLDS = [0]
USTAR_SCENARIOS = ['CUT_NONE']
USTAR_SCENARIO = USTAR_SCENARIOS[0]

# --- Level 4.1: meteo drivers ------------------------------------------------------------------
TA_F = 'TA_T1_47_1_HOMOGENIZED_gfXG'    # degC, complete; also used for ET
SW_IN_F = 'SW_IN_T1_47_1_gfXG'          # W m-2, complete
VPD_F = 'VPD_T1_47_1_HOMOGENIZED'       # kPa, complete
TA_FLAG = 'FLAG_TA_T1_47_1_ISFILLED'    # 0 = measured at 47 m
SW_IN_FLAG = 'FLAG_SW_IN_T1_47_1_ISFILLED'
VPD_FLAG = 'FLAG_VPD_T1_47_1_ISFILLED'
FEATURES = [TA_F, SW_IN_F, VPD_F]

FLUX_UNITS = 'W m-2'

# 30-min LE in W m-2 -> MJ m-2 per half-hour, for sums and cumulatives.
W_TO_MJ = 1800 / 1e6

# MDS gap-filling, next to XGBoost, to compare the two methods. mode='diive' fills every gap,
# also long ones; the QC flags keep model-filled driver values out of the similar samples.
RUN_MDS = True
MDS_SETTINGS = dict(mode='diive', fill_long_gaps=True)

# XGBoost hyperparameters, at the production values suggested by the diive notebook.
XGB_SETTINGS = dict(n_estimators=1000, max_depth=6, learning_rate=0.05,
                    random_state=42, n_jobs=-1, shap_max_rows=5000)

# --- Evapotranspiration ------------------------------------------------------------------------
# et_from_le returns mm h-1; times this factor gives mm per half-hour.
HOURS_PER_RECORD = 0.5

In [ ]:
from datetime import datetime
import importlib.metadata
import warnings

import diive as dv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from diive.core.ml.feature_engineer import FeatureEngineer
from diive.core.plotting.styles import chainplots as cp
from diive.core.times.times import doy_cumulatives_per_year, doy_mean_cumulative
from diive.flux.postprocessingchain import (
    init_flux_data,
    make_level32_detector,
    run_level2,
    run_level31,
    run_level32,
    run_level33_constant_ustar,
    run_level41_mds,
    run_level41_xgb,
    # Reporting over the finished chain
    plot_feature_ranks_per_year,
    plot_tree_selection,
    plot_gapfilling_comparison,
    plot_gapfilling_input,
    plot_quality_heatmaps,
    plot_quality_timeseries,
    report_annual_sums,
    report_gapfilling_comparison,
    report_gapfilling_feature_importances,
    report_gapfilling_input,
    report_gapfilling_model_scores,
    report_gapfilling_poolyears,
    report_gapfilling_variables,
    report_gaps,
    report_model_overview,
    report_traintest_model_scores,
    report_tree_selection,
)
from diive.variables.thermodynamic import et_from_le

# The notebook's inline backend renders each figure itself; diive's own fig.show() call can only
# warn that it cannot open a window.
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')

# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

## 📂 Load data

`init_flux_data` adds `SW_IN_POT`, `DAYTIME` and `NIGHTTIME` itself and stops if they already exist,
so they are removed from the input first.

In [ ]:
df = dv.load_parquet(filepath=SOURCEFILE)
df = df.drop(columns=[c for c in ('SW_IN_POT', 'DAYTIME', 'NIGHTTIME') if c in df.columns])
df = df.loc[df.index.year.isin(YEARS)].copy()
assert sorted(set(df.index.year)) == sorted(YEARS), f"the input lacks some of the years {YEARS}"
assert (df.index[0], df.index[-1]) == (pd.Timestamp(f'{min(YEARS)}-01-01 00:15'),
                                       pd.Timestamp(f'{max(YEARS)}-12-31 23:45')), \
    f"the selected years run from {df.index[0]} to {df.index[-1]}, not over whole calendar years"

for _c in [FLUXCOL, TA_F, SW_IN_F, VPD_F, TA_FLAG, SW_IN_FLAG, VPD_FLAG, *LEVEL2_INPUTS,
           LEVEL2_SETTINGS['signal_strength']['signal_strength_col']]:
    assert _c in df.columns, f"{_c} is not in {SOURCEFILE}"
for _c in FEATURES:
    assert df[_c].notna().all(), f"driver {_c} has gaps"
print(f"{len(df):,} records x {df.shape[1]} columns, {df.index[0]} -> {df.index[-1]}")
print(f"{FLUXCOL}: {df[FLUXCOL].notna().sum():,} records")

## Step 1: set up the data

Calculates the potential radiation, marks day and night, and stores the site coordinates that later
steps use. `*_ACCEPT_QCF_BELOW = 2` keeps records with quality flag 0 (all tests passed) and 1
(minor warnings).

In [ ]:
data = init_flux_data(
    df=df,
    fluxcol=FLUXCOL,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=UTC_OFFSET,
    nighttime_threshold=NIGHTTIME_THRESHOLD,
    daytime_accept_qcf_below=DAYTIME_ACCEPT_QCF_BELOW,
    nighttime_accept_qcf_below=NIGHTTIME_ACCEPT_QCF_BELOW,
)
print(data)

## Step 2: quality flags (Level 2)

The tests in `LEVEL2_SETTINGS`, e.g. steady state and developed turbulence (SSITC), the spectral
correction factor, the signal strength of the analyser and spikes and dropouts in the raw data
(Vickers and Mahrt 1997). The table shows how many records each test removes.

In [ ]:
data = run_level2(data, **LEVEL2_SETTINGS)
print(f'After L2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: Level-2 quality flag over time

In [ ]:
data.levels.level2_qcf.showplot_qcf_heatmaps()

### 📋 Table: result of each Level-2 test

In [ ]:
data.levels.level2_qcf.report_qcf_flags()

## Step 3: no storage term (Level 3.1)

No storage term is added: FLUXNET2015 and ONEFlux take LE as a turbulent flux without storage
correction (Pastorello et al. 2020). Level 3.1 runs with `set_storage_to_zero=True`, because the
later levels need it ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#storage-term)).

In [ ]:
data = run_level31(data, set_storage_to_zero=True)
print(f'After L3.1: {data.filteredseries.dropna().count():,} valid records '
      f'(col: {data.filteredseries.name})')

## Step 4: outlier removal (Level 3.2)

Four tests. Check each plot, then accept the test with `sod.addflag()` in the cell below it.

1. Periods in `REMOVE_DATES`, removed by hand; the reasons are in the site documentation.
2. Absolute limits `ABSLIM`.
3. Hampel filter on double differences: single spikes.
4. Hampel filter per time of day: values wrong for their time of day.

Negative values at night are not tested: LE can be negative at night (dew). How the tests work: [diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#outlier-tests). The table at the end of this step lists
further tests.

In [ ]:
data, sod = make_level32_detector(data)

### 📊 Plot: periods removed by hand

In [ ]:
# Only the date ranges that overlap the processed records; none means no test and no flag.
_first, _last = data.full_df.index[0], data.full_df.index[-1]
_remove = [_d for _d in REMOVE_DATES
           if pd.Timestamp(_d[0]) <= _last and pd.Timestamp(_d[1]) + pd.Timedelta('1D') > _first]
if _remove:
    sod.flag_manualremoval_test(remove_dates=_remove, showplot=True, verbose=True)
else:
    print(f"No manual removal dates in {_first:%Y-%m-%d} - {_last:%Y-%m-%d}; test skipped.")

In [ ]:
if _remove:
    sod.addflag()  # accept the manual removal

### 📊 Plot: absolute limits

In [ ]:
sod.flag_outliers_abslim_test(minval=ABSLIM[0], maxval=ABSLIM[1], showplot=True, verbose=True)

In [ ]:
sod.addflag()  # accept this test

### 📊 Plot: Hampel filter on double differences

In [ ]:
sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=True,
                              separate_day_night=True, showplot=True, verbose=True, repeat=True)

In [ ]:
sod.addflag()  # accept this test

### 📊 Plot: Hampel filter on values per time of day

In [ ]:
sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=False,
                              by_time_of_day=True, time_of_day_slots=1, min_records=10,
                              separate_day_night=True, showplot=True, verbose=True, repeat=True)

In [ ]:
sod.addflag()  # accept this test

### 📋 Table: outlier tests available in diive

Every test of the outlier detector `sod`, with the ones used above marked. `help(sod.<method>)`
shows the settings of a test.

In [ ]:
_used = {'flag_manualremoval_test', 'flag_outliers_abslim_test', 'flag_outliers_hampel_test'}
_rows = []
for _name in sorted(dir(type(sod))):
    if not (_name.startswith('flag_') and _name.endswith('_test')):
        continue
    # The first sentence of the docstring, which may run over two lines.
    _doc = (getattr(type(sod), _name).__doc__ or '').strip().split('\n\n')[0]
    _first = ' '.join(_doc.split()).split('. ')[0].rstrip('.')
    _rows.append({'method': f'sod.{_name}()', 'what it does': _first,
                  'used here': 'yes' if _name in _used else ''})
display(pd.DataFrame(_rows).set_index('method'))

### Apply the Level-3.2 flags

In [ ]:
data = run_level32(data, outlier_detector=sod)
print(f'After L3.2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: all removed outliers

Every record Level 3.2 removed, coloured by the test that removed it, with the share removed per
month and per hour of the day.

In [ ]:
data.levels.level32.showplot_overview();

### 📊 Plot: Level-3.2 quality flag over time

In [ ]:
data.levels.level32_qcf.showplot_qcf_heatmaps()

### 📋 Table: result of each Level-3.2 test

In [ ]:
data.levels.level32_qcf.report_qcf_flags()

## Step 5: no USTAR filter (Level 3.3)

No USTAR filter for LE, as in FLUXNET2015 and ONEFlux (Pastorello et al. 2020). The gap-filling
needs Level 3.3, so it runs one scenario, `CUT_NONE`, with a threshold of 0, which removes no record
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#ustar-filter)).

In [ ]:
data = run_level33_constant_ustar(
    data,
    thresholds=USTAR_THRESHOLDS,
    threshold_labels=USTAR_SCENARIOS,
    showplot=False,
    verbose=True,
)

### Records for gap-filling and highest-quality records

The records the gap-filling starts from (QCF 0 and 1, columns `_QCF`) and the highest-quality
records (QCF 0, columns `_QCF0`) for analyses without gap-filled values
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#gapfilling-input)).

### 📋 Table: records for gap-filling

In [ ]:
report_gapfilling_input(data);

### 📊 Plot: records for gap-filling and highest-quality records

In [ ]:
plot_gapfilling_input(data, units=FLUX_UNITS)

### 📋 Table: gaps

The gaps left after Level 3.3, which the gap-filling has to fill. A long gap lasts at least one day.

In [ ]:
report_gaps(data, long_gap_days=1);

## Step 6: model features

The feature engineer adds lagged values, rolling statistics and time terms to the drivers. It is
built once and passed to XGBoost. `target_col` is only a placeholder.

In [ ]:
engineer = FeatureEngineer(
    target_col='_target_',
    features_lag=[-2, 2],
    features_lag_stepsize=1,
    features_rolling=[2, 4, 12, 24, 48],
    features_rolling_stats=['median', 'min', 'max', 'std'],
    features_ema=[6, 12, 24, 48],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    verbose=0,
)

## Step 7: gap-filling (Level 4.1)

XGBoost fills the gaps of this dataset, MDS for comparison (Step 10).

### XGBoost

One model per year, trained on three years, with features selected by SHAP
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#xgboost-gapfilling)).

The number of trees is chosen once for all models by blocked cross-validation
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#tree-selection)).

In [ ]:
data = run_level41_xgb(
    data,
    features=FEATURES,
    engineer=engineer,
    reduce_features=True,
    n_estimators_cv=True,
    verbose=1,
    **XGB_SETTINGS,
)

### MDS

Marginal distribution sampling (Reichstein et al. 2005) in `mode='diive'`, filling every gap from
measured driver values only ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#mds-gapfilling)).

**Skipped** when `RUN_MDS = False`.

In [ ]:
if RUN_MDS:
    data = run_level41_mds(
        data,
        swin=SW_IN_F, ta=TA_F, vpd=VPD_F,  # W m-2, degC, kPa
        swin_qc=SW_IN_FLAG, ta_qc=TA_FLAG, vpd_qc=VPD_FLAG,
        **MDS_SETTINGS,
    )

## Step 8: results

How many records were measured and how many were filled, then the annual sum of LE in MJ m-2, for
each gap-filling method. The share measured is given in brackets: a year with long gaps rests
largely on the model. An incomplete year is marked.

### 📋 Table: measured and filled records

In [ ]:
report_gapfilling_variables(data)

### 📋 Table: annual sums of LE

In [ ]:
report_annual_sums(data, gain=W_TO_MJ, units='MJ m-2');

## Step 9: model quality

First the scores on test data: a random 25 % of the complete records, which the model did not see in
training. The number of trees is chosen without the test data. Because the test
records lie between training records, the scores describe short gaps better than gaps of several
days.

### 📋 Table: scores on test data

In [ ]:
report_traintest_model_scores(data)

### 📋 Table: gap-filling models per year

How each model was built and how well it did: the years it was trained on, its features, the
training and test records, the number of trees, the records of the final fit,
the test scores, and how the gaps of its year were filled.

In [ ]:
report_model_overview(data);

### 📋 Table: number of trees from blocked cross-validation

The scores on held-out blocks of days at the chosen number of trees, next to the scores on the
random test records. The blocked scores describe gaps of days, the random ones single missing
records ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#tree-selection)).

In [ ]:
report_tree_selection(data);

### 📊 Plot: number of trees from blocked cross-validation

The error on held-out blocks against the number of trees, one line per pool, and the chosen
number. The curves should flatten well before the largest number tested.

In [ ]:
plot_tree_selection(data);

Next the scores on all measured records. The gaps are filled by a model refitted on all of them, so
these scores are better than the test scores and only show how closely the model fits.

### 📋 Table: scores on all measured records

In [ ]:
report_gapfilling_model_scores(data)

### 📋 Table: years used to train each year's model

In [ ]:
report_gapfilling_poolyears(data)

### 📋 Table: most important features (SHAP)

In [ ]:
report_gapfilling_feature_importances(data, ustar_scenario=USTAR_SCENARIO)

### 📊 Plot: feature ranks per year

In [ ]:
plot_feature_ranks_per_year(data)

## Step 10: result plots

### 📊 Plot: LE by quality class, heatmaps, XGBoost

The highest-quality records (QCF 0), the records the gap-filling used (QCF 0 and 1), the rejected
records, and the gap-filled result, on one colour scale.

In [ ]:
plot_quality_heatmaps(data, ustar_scenario=USTAR_SCENARIO, units=FLUX_UNITS, method='xgb')

### 📊 Plot: LE by quality class, day and night, XGBoost

Daytime and nighttime records by quality class, then the gap-filled result, with measured and
filled records shown apart.

In [ ]:
plot_quality_timeseries(data, ustar_scenario=USTAR_SCENARIO, units=FLUX_UNITS, method='xgb')

### 📊 Plot: measured and gap-filled LE

In [ ]:
data.plot_gapfilled_heatmaps(ustar_scenario=USTAR_SCENARIO, units=FLUX_UNITS)

### 📊 Plot: cumulative LE

One line per gap-filling method, and the measured records alone (gaps counted as zero).

In [ ]:
data.plot_cumulative_comparison(
    ustar_scenario=USTAR_SCENARIO,
    conv_factor=W_TO_MJ,
    units='MJ m-2',
)

### 📊 Plot: cumulative LE per year

One panel per gap-filling method, on the same y-axis. One line per year; the band is the mean ± 1 SD
of the full years. An incomplete last year is left out of the band.

In [ ]:
# One panel per gap-filling method, on a shared y-axis. Years run from light to dark, so the most
# recent year is the darkest line.
_gf = data.gapfilled_cols()
_names = {'mds': 'MDS', 'xgb': 'XGBoost', 'rf': 'random forest'}
_months = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
# A last year that ends before 31 December stays out of the mean and the band.
_end = data.fpc_df.index[-1]
_partial = [_end.year] if (_end.month, _end.day) != (12, 31) else None
fig, axes = plt.subplots(1, len(_gf), figsize=(8 * len(_gf), 6.6), sharex=True, sharey=True,
                         layout='constrained', squeeze=False)
for _ax, (_method, _cols) in zip(axes[0], _gf.items()):
    _cum = doy_cumulatives_per_year(data.fpc_df[_cols[USTAR_SCENARIO]] * W_TO_MJ)
    _cum = _cum.loc[:, _cum.notna().any()]
    _years = list(_cum.columns)
    _shades = plt.get_cmap(cp.METHOD_CMAPS.get(_method, 'Greys'))(
        np.linspace(0.35, 1.0, len(_years)) if len(_years) > 1 else [0.85])
    if len(_years) > 1:
        _ref = doy_mean_cumulative(cumulatives_per_year_df=_cum,
                                   excl_years_from_reference=_partial)
        # Day 366 exists only in leap years, so the band stops before it.
        _ref = _ref.loc[_ref.index < 366]
        # The legend gives the mean and SD of the annual totals of the full years.
        _totals = [_cum[_y].dropna().iloc[-1] for _y in _years if _y not in (_partial or [])]
        _end_mean, _end_sd = np.mean(_totals), np.std(_totals, ddof=1)
        _ax.fill_between(_ref.index, _ref['MEAN-SD'], _ref['MEAN+SD'], color=cp.KEPT,
                         alpha=0.3, linewidth=0, label=f'mean ± 1 SD: ±{_end_sd:,.0f}')
        _ax.plot(_ref.index, _ref['MEAN_DOY_TIME'], color=cp.INK, linewidth=1.2,
                 linestyle=(0, (4, 3)), label=f'mean: {_end_mean:,.0f}')
    for _shade, _year in zip(_shades, _years):
        _valid = _cum[_year].dropna()
        _ax.plot(_cum.index, _cum[_year], color=_shade, linewidth=1.5,
                 label=f"{_year}: {_valid.iloc[-1]:,.0f}")
    _ax.axhline(0, color=cp.INK, linewidth=0.8, alpha=0.5)
    cp.style_axes(_ax)
    cp.panel_label(_ax, _names.get(_method, _method))
    _ax.set_xticks(_months, ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    _ax.set_xlim(1, 367)
    _ax.legend(frameon=False, fontsize=8, ncol=1 if len(_years) < 8 else 2)
axes[0][0].set_ylabel('cumulative LE (MJ m-2)')
_subtitle = "One panel per gap-filling method; the total of each year in the legend."
if _partial:
    _subtitle += f" {_end.year} ends on {_end:%d %B} and is left out of the mean and the band."
cp.figure_title(fig, 'Cumulative LE per year', _subtitle)
plt.show()

### Gap-filling methods compared

The difference between the sums of MDS and XGBoost comes from the filled gaps alone; the table
splits it by day and night and by gap length ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#method-comparison)).

**Skipped** when `RUN_MDS = False`.

#### 📋 Table: MDS and XGBoost, sums and their difference

In [ ]:
if RUN_MDS:
    report_gapfilling_comparison(data, methods=('mds', 'xgb'), gain=W_TO_MJ, units='MJ m-2');

#### 📊 Plot: where MDS and XGBoost differ

The filled values of both methods at the same gaps, their mean difference by month and hour, the
contribution of each month to the difference of the sums, and the cumulative difference.

In [ ]:
if RUN_MDS:
    plot_gapfilling_comparison(data, ustar_scenario=USTAR_SCENARIO, methods=('mds', 'xgb'),
                               gain=W_TO_MJ, units='MJ m-2', flux_units=FLUX_UNITS);

## Step 11: evapotranspiration

ET is LE divided by the latent heat of vaporization, which depends on air temperature
(`et_from_le` with `TA_F`, after the R package bigleaf). The ET column of each gap-filling method
holds mm per half-hour and sums directly to totals. The input column `ET` is the EddyPro rate of
the measured records only.

### 📋 Table: ET columns

In [ ]:
_ta = data.full_df[TA_F].reindex(data.fpc_df.index)
assert _ta.notna().all(), f"{TA_F} has gaps"
ET_COLS = {}  # {method: ET column}
_et = {}
for _method, _cols in data.gapfilled_cols().items():
    _le = data.fpc_df[_cols[USTAR_SCENARIO]]
    assert _le.name.startswith(FLUXCOL), _le.name
    _name = 'ET' + _le.name[len(FLUXCOL):]
    _et[_name] = et_from_le(le=_le, ta=_ta) * HOURS_PER_RECORD  # mm h-1 -> mm per half-hour
    ET_COLS[_method] = _name
ET_EXPORT = pd.DataFrame(_et)
assert ET_EXPORT.notna().all().all(), "a gap-filled record has no ET"

_names = {'mds': 'MDS', 'xgb': 'XGBoost', 'rf': 'random forest'}
display(pd.DataFrame({
    'from LE column': [data.gapfilled_cols()[_k][USTAR_SCENARIO] for _k in ET_COLS],
    'method': [_names.get(_k, _k) for _k in ET_COLS],
    'units': 'mm per half-hour',
    'records': [int(ET_EXPORT[_c].count()) for _c in ET_COLS.values()],
    'mean (mm d-1)': [round(float(ET_EXPORT[_c].mean()) * 48, 2) for _c in ET_COLS.values()],
}, index=pd.Index(list(ET_COLS.values()), name='ET column')))

### 📋 Table: annual sums of ET

Annual sums in mm for each gap-filling method, and their difference. An incomplete last year is a
partial sum.

In [ ]:
ANNUAL_ET = ET_EXPORT.groupby(ET_EXPORT.index.year).sum()
ANNUAL_ET.columns = [_names.get(_k, _k) for _k in ET_COLS]
ANNUAL_ET.index.name = 'year'
if {'XGBoost', 'MDS'} <= set(ANNUAL_ET.columns):
    ANNUAL_ET['XGBoost - MDS'] = ANNUAL_ET['XGBoost'] - ANNUAL_ET['MDS']
print("Annual sums of ET (mm)")
display(ANNUAL_ET.round(1))

### 📊 Plot: annual sums of ET

In [ ]:
_colors = {_names.get(_k, _k): _v[1] for _k, _v in cp.METHOD_COLORS.items()}  # hue of each method
_methods = [_names.get(_k, _k) for _k in ET_COLS]
_years = ANNUAL_ET.index
_width = 0.8 / len(_methods)
_end = ET_EXPORT.index[-1]
fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(_years) + 4), 5), layout='constrained')
for _i, _method in enumerate(_methods):
    _x = np.arange(len(_years)) + (_i - (len(_methods) - 1) / 2) * _width
    ax.bar(_x, ANNUAL_ET[_method], _width, color=_colors.get(_method, cp.KEPT), label=_method)
    for _xi, _v in zip(_x, ANNUAL_ET[_method]):
        ax.annotate(f'{_v:.0f}', (_xi, _v), ha='center', va='bottom', fontsize=7, color=cp.INK,
                    xytext=(0, 2), textcoords='offset points')
ax.set_xticks(np.arange(len(_years)), _years)
ax.set_ylabel('annual ET (mm)')
cp.style_axes(ax)
fig.legend(*ax.get_legend_handles_labels(), loc='outside lower center', frameon=False,
           ncol=len(_methods), fontsize=9)
cp.figure_title(fig, 'Annual sums of ET',
                "One bar per gap-filling method."
                + (f" {_end.year} ends on {_end:%d %B}." if (_end.month, _end.day) != (12, 31)
                   else ""))
plt.show()

## 💾 Step 12: export

The output file holds only what this notebook adds: every column the chain created and the ET
columns of Step 11. The input columns stay unchanged in the input file; the merge into the fullset
joins the outputs of the flux notebooks with it on the timestamp.

In [ ]:
# Only the new columns: the input columns are in the input file and are added back in the merge.
_all = data.merged_df().join(ET_EXPORT)
out_df = _all[[c for c in _all.columns if c not in df.columns]]
assert out_df.index.equals(df.index), "the output timestamps differ from the input"
assert set(ET_EXPORT.columns) <= set(out_df.columns), "ET columns missing from the output"
print(f"ET columns in the output: {list(ET_EXPORT.columns)}")
print(f"{out_df.shape[0]:,} records x {out_df.shape[1]} columns (input columns left out: "
      f"{_all.shape[1] - out_df.shape[1]})")
filepath = dv.save_parquet(data=out_df, filename=OUTNAME, outpath=OUTPATH)

***

# End of notebook.

In [ ]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")